In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob, json as _json, joblib
from pathlib import Path
from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    import japanize_matplotlib 
except ImportError:
    _jp_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'MS PGothic',
                 'IPAexGothic', 'Noto Sans CJK JP']
    import matplotlib.font_manager as _fm
    _available = {f.name for f in _fm.fontManager.ttflist}
    _matched = [f for f in _jp_fonts if f in _available]
    if _matched:
        matplotlib.rcParams['font.family'] = 'sans-serif'
        matplotlib.rcParams['font.sans-serif'] = _matched + list(matplotlib.rcParams['font.sans-serif'])
matplotlib.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [13]:
# ─── Configuration ────────────────────────────────────────────────────────────

RESULT_DIR = f"./result_dir/XXXXXX_XXXX_dino" # train folder name 
print(f'RESULT_DIR: {RESULT_DIR}')

with open(os.path.join(RESULT_DIR, 'meta.json')) as f:
    meta = _json.load(f)

model_name = meta['model_name']
n_classes  = meta['n_classes']
print(f'model_name : {model_name}')
print(f'n_classes  : {n_classes}')

CLASS_NAMES = None


RESULT_DIR: ./result_dir/261106_2034_dino
model_name : culture-noAug
n_classes  : 4


In [14]:
# ─── Load all saved artifacts ─────────────────────────────────────────────────
features   = np.load(os.path.join(RESULT_DIR, f'{model_name}_dino_features.npy'))
print(f'features    : {features.shape}')

y_all      = np.load(os.path.join(RESULT_DIR, f'{model_name}_y_all.npy'))
images_all = np.load(os.path.join(RESULT_DIR, f'{model_name}_images_all.npy'))
print(f'y_all       : {y_all.shape}')
print(f'images_all  : {images_all.shape}')

scaler_sk   = joblib.load(os.path.join(RESULT_DIR, f'{model_name}_scaler.pkl'))
feat_scaled = scaler_sk.transform(features)
print('Scaler loaded.')

# ── Teacher backbone ───────────────────────────────────────────────
FEAT_DIM = features.shape[1]

HEAD_HIDDEN  = 512
HEAD_BOTTLE  = meta.get('HEAD_BOTTLE', 128)
OUT_DIM      = meta.get('OUT_DIM', 1024)
print(f'HEAD_BOTTLE={HEAD_BOTTLE},  OUT_DIM={OUT_DIM}')

class CNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        def block(ci, co, pool=True):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(True),
                      nn.BatchNorm2d(co),
                      nn.Conv2d(co, co, 3, padding=1), nn.ReLU(True),
                      nn.BatchNorm2d(co)]
            if pool: layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)
        self.encoder = nn.Sequential(
            block(3,64), block(64,128), block(128,256),
            block(256,512), block(512,512))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(True),
            nn.Linear(256, FEAT_DIM))
    def forward(self, x):
        return self.fc(self.pool(self.encoder(x)))

ckpt = torch.load(os.path.join(RESULT_DIR, f'{model_name}_dino_checkpoint.pt'), map_location=device, weights_only=False)
teacher_backbone = CNNBackbone().to(device)
teacher_backbone.load_state_dict(ckpt['teacher_backbone_state'])
teacher_backbone.eval()
print('Teacher backbone loaded.')

if 'loss_history' in ckpt:
    loss_history = ckpt['loss_history']
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(1, len(loss_history)+1), loss_history,
            color='royalblue', linewidth=1.6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('DINO Loss')
    ax.set_title('DINO Pre-training Loss (v2)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'dino_loss_curve_test.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Final loss: {loss_history[-1]:.4f}')


features    : (3560, 128)
y_all       : (3560,)
images_all  : (3560, 150, 150, 1)
Scaler loaded.
HEAD_BOTTLE=128,  OUT_DIM=256
Teacher backbone loaded.
Final loss: 2.4738


C:\TEMP\ipykernel_20384\2520730994.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ─── K: Elbow ────────────────────────────────────────────────────────

from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix as sk_cm

K_range  = range(1, 16)
inertias = []

for k in tqdm(K_range, desc='K-means'):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    km.fit(feat_scaled)
    inertias.append(km.inertia_)

try:
    from kneed import KneeLocator
    kneedle = KneeLocator(list(K_range), inertias,
                          curve='convex', direction='decreasing')
    k_elbow = kneedle.elbow or 3
except ImportError:
    diffs2  = np.diff(inertias, 2)
    k_elbow = int(np.argmax(diffs2) + 2)

print(f'Elbow → K = {k_elbow}')

best_K = k_elbow
print(f'\nUsing K = {best_K}  (Elbow)')

# ── K-Means ───────────────────────────────────────────────────────────────
km_final = KMeans(n_clusters=best_K, n_init=20, random_state=SEED)
kmeans_labels = km_final.fit_predict(feat_scaled)
print(f'Cluster sizes: {dict(zip(*np.unique(kmeans_labels, return_counts=True)))}')

if CLASS_NAMES is None:
    cls_names = [str(i) for i in range(n_classes)]
else:
    cls_names = list(CLASS_NAMES)

cluster_labels = [f'Cluster {k}' for k in range(best_K)]

# ─── 5-fold CV ──────────────────────────────────────────────

n_splits = 5
cm_accum = np.zeros((best_K, best_K), dtype=float)

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
for train_idx, test_idx in skf.split(feat_scaled, kmeans_labels):
    X_tr, X_te = feat_scaled[train_idx], feat_scaled[test_idx]
    y_true_te  = kmeans_labels[test_idx]      

    km_cv = KMeans(n_clusters=best_K, n_init=10, random_state=SEED)
    km_cv.fit(X_tr)
    y_pred_te = km_cv.predict(X_te)        

    # Hungarian
    cm_raw = sk_cm(y_true_te, y_pred_te, labels=list(range(best_K)))
    _, col_ind = linear_sum_assignment(-cm_raw)
    label_map  = {c: r for r, c in enumerate(col_ind)}
    y_pred_mapped = np.array([label_map.get(l, l) for l in y_pred_te])

    cm_accum += sk_cm(y_true_te, y_pred_mapped, labels=list(range(best_K)))

cm_pct = cm_accum / cm_accum.sum(axis=1, keepdims=True) * 100

# ─── K-Means ──────────────────────
comp = np.zeros((best_K, n_classes), dtype=float)
for k in range(best_K):
    mask  = kmeans_labels == k
    total = mask.sum()
    if total > 0:
        for c in range(n_classes):
            comp[k, c] = (y_all[mask] == c).sum() / total * 100

sns.set_style('whitegrid')

fig = plt.figure(figsize=(14, 10))


ax_a = fig.add_subplot(2, 2, 1)
ax_a.plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=1.8, markersize=5)
ax_a.axvline(k_elbow, color='tomato', linestyle='--',
             label=f'Elbow K={k_elbow}', linewidth=1.5)
ax_a.set_xlabel('K', fontsize=11)
ax_a.set_ylabel('Inertia', fontsize=11)
ax_a.legend(fontsize=10)
ax_a.grid(True, alpha=0.3)
ax_a.text(-0.12, 1.05, 'A', transform=ax_a.transAxes,
          fontsize=14, fontweight='bold')


ax_b = fig.add_subplot(2, 2, 2)
sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=cluster_labels, yticklabels=cluster_labels,
            vmin=0, vmax=100, ax=ax_b, cbar_kws={'label': ''})
ax_b.set_xlabel('Predict classes', fontsize=11)
ax_b.set_ylabel('True classes', fontsize=11)
ax_b.set_xticklabels(ax_b.get_xticklabels(), rotation=45, ha='right')
ax_b.set_yticklabels(ax_b.get_yticklabels(), rotation=0)
ax_b.text(-0.18, 1.05, 'B', transform=ax_b.transAxes,
          fontsize=14, fontweight='bold')


ax_c = fig.add_subplot(2, 2, 3)
sns.heatmap(comp, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=cls_names, yticklabels=cluster_labels,
            vmin=0, vmax=100, ax=ax_c, cbar_kws={'label': ''})
ax_c.set_xlabel('Cell type', fontsize=11)
ax_c.set_ylabel('K-means cluster', fontsize=11)
ax_c.set_xticklabels(ax_c.get_xticklabels(), rotation=45, ha='right')
ax_c.set_yticklabels(ax_c.get_yticklabels(), rotation=0)
ax_c.text(-0.18, 1.05, 'C', transform=ax_c.transAxes,
          fontsize=14, fontweight='bold')

plt.tight_layout()
out_path = os.path.join(RESULT_DIR, 'Figure.tif')
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')


K-means:   0%|          | 0/15 [00:00<?, ?it/s]

Elbow → K = 4

Using K = 4  (Elbow)
Cluster sizes: {0: 411, 1: 934, 2: 1113, 3: 1102}
